In [ ]:
# historic bird data from eBird
import time
from datetime import date, timedelta
import pandas as pd
import requests

API_KEY = ""
HEADERS = {"X-eBirdApiToken": API_KEY}
SPECIES_CODE = "mallar3" 
RESULTS = [] 

# 20 european countries
EUROPEAN_COUNTRIES = {
    'Netherlands': 'NL',
    'Germany': 'DE',
    'France': 'FR',
    'United Kingdom': 'GB',
    'Sweden': 'SE',
    'Denmark': 'DK',
    'Poland': 'PL',
    'Spain': 'ES',
    'Italy': 'IT',
    'Switzerland': 'CH',
    'Austria': 'AT',
    'Belgium': 'BE',
    'Norway': 'NO',
    'Finland': 'FI',
    'Ireland': 'IE',
    'Portugal': 'PT',
    'Czech Republic': 'CZ',
    'Hungary': 'HU',
    'Croatia': 'HR',
    'Greece': 'GR'
}


def get_data(country_name, region_code, current_date):
    """
    Getting bird observations for a region and date from eBird.
    
    Args:
        country_name (str): Full name of the country.
        region_code (str): Two-letter country code for eBird.
        current_date (date): A datetime.date object.
    """

    # # Convert date object to YYYY/MM/DD
    date_str = current_date.strftime("%Y/%m/%d")
    url = f"https://api.ebird.org/v2/data/obs/{region_code}/historic/{date_str}"
    params = {
        "includeProvisional": "true",
        "hotspot": "false"
    }

    print(f"Start request for {region_code} on {date_str}...")
    
    try:
        # Do the get request with a 10 second timeout
        response = requests.get(url, headers=HEADERS, params=params, timeout=10)
        # Checking, if the request runs withot errors
        if response.status_code == 200:
            data = response.json()

            # Process each observation record in the response list
            for obs in data:
                if obs.get("speciesCode") == SPECIES_CODE:
                    
                    # Extracting the actual bird count
                    raw_count = obs.get("howMany", 1)
                    try:
                        count_value = int(raw_count)
                    except (ValueError, TypeError):
                        count_value = 1

                    # Append a dictionary for easy conversion to a Pandas DataFrame
                    RESULTS.append({
                        "Date": date_str,
                        "Country": country_name,
                        "SpeciesCode": SPECIES_CODE,
                        "DuckCount": count_value
                    })
            
            print(f"Added observations for {region_code}")
        else:
            # Giving out the error
            print(f"Error {response.status_code} for {region_code} on {date_str}")
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")


# executing our function
if __name__ == "__main__":
    # Defining the time period for the analysis
    years = [2020, 2021, 2022, 2023, 2024]

    # Iterate through each country
    for country_name, country_code in EUROPEAN_COUNTRIES.items():
        print(f"\nAnalysiere Land: {country_name}")
        
        # Iterate through each year
        for year in years:
            start_date = date(year, 3, 1)
           
           # Iterate through all 31 days of March
            for day in range(31):
                ## Calculate the exact target date
                target_date = start_date + timedelta(days=day)
                get_data(country_name, country_code, target_date)
                
                # Short pause to not overload the API
                time.sleep(0.2) 

    # Creating our data frame and saving it
    if RESULTS:
        # Convert the list of dictionaries into a structured DataFrame
        df = pd.DataFrame(RESULTS)
        file_name = "europe_ducks_march_2020_2024.csv"
        df.to_csv(file_name, index=False)
        
        print(f"\nData saved as: {file_name}")
        print(df.head())
    else:
        print("\nKeine Daten gefunden.")

    print("------- DONE -------")



Analysiere Land: Netherlands
Start request for NL on 2020/03/01...
1 duck observations added for NL on 2020/03/01
Start request for NL on 2020/03/02...
1 duck observations added for NL on 2020/03/02
Start request for NL on 2020/03/03...
1 duck observations added for NL on 2020/03/03
Start request for NL on 2020/03/04...
1 duck observations added for NL on 2020/03/04
Start request for NL on 2020/03/05...
1 duck observations added for NL on 2020/03/05
Start request for NL on 2020/03/06...
1 duck observations added for NL on 2020/03/06
Start request for NL on 2020/03/07...
1 duck observations added for NL on 2020/03/07
Start request for NL on 2020/03/08...
1 duck observations added for NL on 2020/03/08
Start request for NL on 2020/03/09...
1 duck observations added for NL on 2020/03/09
Start request for NL on 2020/03/10...
1 duck observations added for NL on 2020/03/10
Start request for NL on 2020/03/11...
1 duck observations added for NL on 2020/03/11
Start request for NL on 2020/03/12.

In [ ]:
import time
from datetime import date, timedelta
import pandas as pd
import requests

API_KEY = ''
HEADERS = {'X-eBirdApiToken': API_KEY}
SPECIES_CODE = 'mallar3' 
RESULTS = []

# 20 european countries
EUROPEAN_COUNTRIES = {
    'Netherlands': 'NL',
    'Germany': 'DE',
    'France': 'FR',
    'United Kingdom': 'GB',
    'Sweden': 'SE',
    'Denmark': 'DK',
    'Poland': 'PL',
    'Spain': 'ES',
    'Italy': 'IT',
    'Switzerland': 'CH',
    'Austria': 'AT',
    'Belgium': 'BE',
    'Norway': 'NO',
    'Finland': 'FI',
    'Ireland': 'IE',
    'Portugal': 'PT',
    'Czech Republic': 'CZ',
    'Hungary': 'HU',
    'Croatia': 'HR',
    'Greece': 'GR'
}


def get_recent_data(country_name, country_code):
    """
    Getting bird observations for a region for the last 30 days.
    
    Args:
        country_name (str): Full name of the country.
        region_code (str): Two-letter country code for eBird.
    """
    
    url = f"https://api.ebird.org/v2/data/obs/{country_code}/recent/{SPECIES_CODE}"
    params = {
        "back": 30,  # last 30 days
        "includeProvisional": "true"
    }

    try:
        # Do the get request with a 10 second timeout
        response = requests.get(url, headers=HEADERS, params=params, timeout=10)
        # Checking, if the request runs withot errors
        if response.status_code == 200:
            observations = response.json()

            date_dict = {}

            # Process each observation record in the response list
            for obs in observations:
                obs_date = obs.get('obsDt', '').split(' ')[0]
                if obs_date not in date_dict:
                    date_dict[obs_date] = []
                date_dict[obs_date].append(obs)

            # Aggregate the observations per date
            for obs_date, obs_list in date_dict.items():
                total_ducks = 0
                locations = set()

                for obs in obs_list:
                    count = obs.get("howMany", 1)
                    # making sure, that count is an int
                    try:
                        total_ducks += int(count)
                    except (ValueError, TypeError):
                        total_ducks += 1

                    loc = obs.get("locName")
                    if loc:
                        locations.add(loc)
                # Append a dictionary for easy conversion to DataFrame
                RESULTS.append({
                    "Date": obs_date,
                    "Country": country_name,
                    "DuckCount": total_ducks,
                    "LocationCount": len(locations)
                })
            
            print(f"Erfolgreich: {len(date_dict)} Tage für {country_name} verarbeitet.")

        else:
            print(f"Error {response.status_code} for {country_name}")
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {country_name}: {e}")


# executing our code
if __name__ == "__main__":
    # Iterate through all countrys
    for name, code in EUROPEAN_COUNTRIES.items():
        print(f"Analysiere Land: {name}...")
        get_recent_data(name, code)
        
        # Short pause to not overload our API
        time.sleep(0.5) 

    # Creating our data frame and saving it
    if RESULTS:
        df = pd.DataFrame(RESULTS)
        output_file = "europe_ducks_recent_daily.csv"
        df.to_csv(output_file, index=False)

        print(f"\nSaved as {output_file}")
        print(df.head())
    else:
        print("\nKeine Daten zum Speichern gefunden.")

    print("------ DONE ------")



Analysiere Land: Netherlands

Analysiere Land: Germany

Analysiere Land: France

Analysiere Land: United Kingdom

Analysiere Land: Sweden

Analysiere Land: Denmark

Analysiere Land: Poland

Analysiere Land: Spain

Analysiere Land: Italy

Analysiere Land: Switzerland

Analysiere Land: Austria

Analysiere Land: Belgium

Analysiere Land: Norway

Analysiere Land: Finland

Analysiere Land: Ireland

Analysiere Land: Portugal

Analysiere Land: Czech Republic

Analysiere Land: Hungary

Analysiere Land: Croatia

Analysiere Land: Greece

Saved as europe_ducks_recent_daily.csv
         Date      Country  DuckCount  LocationCount
0  2026-03-04  Netherlands         84             12
1  2026-03-03  Netherlands        706             59
2  2026-03-02  Netherlands        304             49
3  2026-03-01  Netherlands        967             98
4  2026-02-28  Netherlands        220             34
------DONE------


In [ ]:
# Choropleth Plot without Interaction
import pandas as pd
import plotly.express as px

# Load the csv file of recent duck observations
df = pd.read_csv(r"C:\Users\ThinkPad\Documents\Data Science Projekt\RQ 5\europe_ducks_recent_daily.csv")

# Calculate the average for each country
df_country = df.groupby("Country").mean(numeric_only=True).reset_index()

# Calculate the density for each country
df_country["Density"] = df_country["DuckCount"] / df_country["LocationCount"]


def create_choropleth_map(df_country):
    # Creating the Choropleth map
    fig = px.choropleth(
        df_country,
        locations="Country",
        locationmode="country names",
        color="Density",
        hover_name="Country",
        color_continuous_scale="Reds",
        scope="europe"
    )

    # Making the layout and making countries withot data grey
    fig.update_layout(
        title="Mallard-Duck Observation Density in Europe",
        geo=dict(
            showframe=False,
            showcoastlines=True,
            bgcolor="white",
            landcolor="lightgrey"  
        )
    )
    return fig

fig = create_choropleth_map(df_country)
fig.show()

C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_7172\1117931253.py:16: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



In [ ]:
# Chloreopleth Map with interaction
import pandas as pd
import plotly.express as px

# paths for our csv files
PATH_HIST = (
    r"C:\Users\ThinkPad\Documents\Data Science Projekt\RQ 5"
    r"\europe_ducks_march_2020_2024.csv"
)
PATH_DENSITY = (
    r"C:\Users\ThinkPad\Documents\Data Science Projekt\RQ 5"
    r"\europe_ducks_recent_daily.csv"
)


def prepare_map_data(hist_path, density_path):
    # Loading the first csv file
    df_h = pd.read_csv(hist_path)
    df_h["Date"] = pd.to_datetime(df_h["Date"])

    # Calculate dayly ducks per country
    daily_ducks = df_h.groupby(["Country", "Date"])["DuckCount"].sum().reset_index()

    # Calculate the Stability per country
    stability = daily_ducks.groupby("Country")["DuckCount"].agg(
        mean="mean",
        std="std"
    ).reset_index()
    stability["CV"] = stability["std"] / stability["mean"]

    # Loading the second csv file and calculate the density per country
    df_d = pd.read_csv(density_path)
    density_country = df_d.groupby("Country").mean(numeric_only=True).reset_index()
    density_country["Density"] = (
        density_country["DuckCount"] / density_country["LocationCount"]
    )

    # Merge the stability and density data
    return pd.merge(density_country, stability, on="Country", how="left")


def create_interactive_map(df_map):
    # Creating our interactive chloropleth map with interaction
    fig = px.choropleth(
        df_map,
        locations="Country",
        locationmode="country names",
        color="Density",
        hover_name="Country",
        hover_data=["Density", "CV"],
        color_continuous_scale="Reds",
        scope="europe"
    )

    # Basic Layout and making countrys without data grey
    fig.update_layout(
        title="<b>Recent Duck Density and Stability (5 Years) in Europe</b>",
        template="plotly_white",
        geo=dict(
            showframe=False,
            landcolor="lightgrey",
            coastlinecolor="white"
        )
    )

    # Creating the dropdown menu
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=[
                    # The density option
                    dict(
                        label="Density",
                        method="restyle",
                        args=[{
                            "z": [df_map["Density"]],
                            "colorbar.title.text": "Density"
                        }]
                    ),
                    # The Instability option
                    dict(
                        label="Instability (CV)",
                        method="restyle",
                        args=[{
                            "z": [df_map["CV"]],
                            "colorbar.title.text": "Instability (CV)"
                        }]
                    )
                ],
                direction="down",
                showactive=True,
                x=0.05,
                y=1.1
            )
        ]
    )

    return fig


if __name__ == "__main__":
    # Getting the data
    merged_data = prepare_map_data(PATH_HIST, PATH_DENSITY)

    # Executing our visiolazation
    final_map = create_interactive_map(merged_data)
    final_map.show()

C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_7172\364811948.py:44: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



In [ ]:
# Scatterplot without Interaction
import pandas as pd
import plotly.express as px

# Loading data for density
df_density = pd.read_csv(r"C:\Users\ThinkPad\Documents\Data Science Projekt\RQ 5\europe_ducks_recent_daily.csv")
density_country = df_density.groupby("Country").mean(numeric_only=True).reset_index()
density_country["Density"] = density_country["DuckCount"] / density_country["LocationCount"]

# Loading data for stability
df_hist = pd.read_csv(r"C:\Users\ThinkPad\Documents\Data Science Projekt\RQ 5\europe_ducks_march_2020_2024.csv")
df_hist["Date"] = pd.to_datetime(df_hist["Date"])

# Calculating the stability
daily_ducks = df_hist.groupby(["Country","Date"])["DuckCount"].sum().reset_index()
stability = daily_ducks.groupby("Country")["DuckCount"].agg(
    mean="mean",
    std="std"
).reset_index()
stability["CV"] = stability["std"] / stability["mean"]

# Merging the density and stability data
df = pd.merge(density_country, stability, on="Country", how="left")

# Claculate the median lines
median_density = df["Density"].median()
median_cv = df["CV"].median()



# Classify the countrys into the 4 categories
def classify(row):
    
    if row["Density"] >= median_density and row["CV"] <= median_cv:
        return "Stable Core Population"
    
    elif row["Density"] >= median_density and row["CV"] > median_cv:
        return "Seasonal Hotspot"
    
    elif row["Density"] < median_density and row["CV"] <= median_cv:
        return "Small Stable Population"
    
    else:
        return "Random Observations"

df["Category"] = df.apply(classify, axis=1)


# Creating the Scatter Plot
fig = px.scatter(
    df,
    x="Density",
    y="CV",
    color="Category",
    text="Country",
    size="mean",
    hover_name="Country",
    hover_data=["Density","CV","mean","std"],
    title="Recent Duck Density vs Stability over 5 years"
)

# Putting in the country labels
fig.update_traces(textposition="top center")

# Putting in the median lines
fig.add_vline(x=median_density, line_dash="dash")
fig.add_hline(y=median_cv, line_dash="dash")

# Adding the 4 labels into the corners
def add_label(x, y, text):
    fig.add_annotation(
        x=x, y=y, xref="paper", yref="paper",
        text=text, showarrow=False, font=dict(size=12, color="grey"),
        opacity=0.7
    )

add_label(1, 0, "Stable & Dense")      # bottom right
add_label(0, 0, "Stable & Sparse")     # bottom left
add_label(1, 1, "Unstable & Dense")    # top right
add_label(0, 1, "Unstable & Sparse")   # top left

# Giving the axises titles
fig.update_layout(
    xaxis_title="Duck Density",
    yaxis_title="Instability",
)

fig.show()


In [ ]:
# Scatterplot with interaction
import pandas as pd
import plotly.express as px

# Loading the density data
df_density = pd.read_csv(r"C:\Users\ThinkPad\Documents\Data Science Projekt\RQ 5\europe_ducks_recent_daily.csv")
density_country = df_density.groupby("Country").mean(numeric_only=True).reset_index()
density_country["Density"] = density_country["DuckCount"] / density_country["LocationCount"]

# Loading historc data (for stability)
df_hist = pd.read_csv(r"C:\Users\ThinkPad\Documents\Data Science Projekt\RQ 5\europe_ducks_march_2020_2024.csv")
df_hist["Date"] = pd.to_datetime(df_hist["Date"])
daily_ducks = df_hist.groupby(["Country","Date"])["DuckCount"].sum().reset_index()

# Calculating the stability
stability = daily_ducks.groupby("Country")["DuckCount"].agg(
    mean="mean",
    std="std"
).reset_index()
stability["CV"] = stability["std"] / stability["mean"]

# Merge the density and stability data
df = pd.merge(density_country, stability, on="Country", how="left")

# Calculating the median lines
median_density = df["Density"].median()
median_cv = df["CV"].median()



# classify the countrys into the 4 categories
def classify(row):
    
    if row["Density"] >= median_density and row["CV"] <= median_cv:
        return "Stable Core Population"
    
    elif row["Density"] >= median_density and row["CV"] > median_cv:
        return "Seasonal Hotspot"
    
    elif row["Density"] < median_density and row["CV"] <= median_cv:
        return "Small Stable Population"
    
    else:
        return "Random Observations"

df["Category"] = df.apply(classify, axis=1)

# Creating the Scatterplot
fig = px.scatter(
    df,
    x="Density",
    y="CV",
    color="Category",
    text="Country",
    size="mean",
    hover_name="Country",
    hover_data=["Density","CV","mean","std"],
    title="Recent Duck Density vs Stability over 5 years"
)

# Adding the country labels
fig.update_traces(textposition="top center")

# Adding the median lines
fig.add_vline(x=median_density, line_dash="dash")
fig.add_hline(y=median_cv, line_dash="dash")

# Giving the titles to our axises
fig.update_layout(
    xaxis_title="Duck Density",
    yaxis_title="Instability (CV)", 
    template="plotly_white"
)

# Adding the 4 labels into the corners
def add_label(x, y, text):
    fig.add_annotation(
        x=x, y=y, xref="paper", yref="paper",
        text=text, showarrow=False, font=dict(size=12, color="grey"),
        opacity=0.7
    )

add_label(1, 0, "Stable & Dense")      # bottom right
add_label(0, 0, "Stable & Sparse")     # bottom left
add_label(1, 1, "Unstable & Dense")    # top right
add_label(0, 1, "Unstable & Sparse")   # top left


# Calculating the max values for the zoom
x_max = df["Density"].max() * 1.1
y_max = df["CV"].max() * 1.1

# Making the interactive options
fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.01,
            y=1.15,
            showactive=True,
            buttons=[
                # Show everything
                dict(
                    label="Show All Regions",
                    method="relayout",
                    args=[{"xaxis.range": [0, x_max], "yaxis.range": [0, y_max]}]
                ),
                # Show bottom right quadrant
                dict(
                    label="Stable & Dense",
                    method="relayout",
                    args=[{"xaxis.range": [median_density, x_max], "yaxis.range": [0, median_cv]}]
                ),
                # Show top right quadrant
                dict(
                    label="Unstable & Dense",
                    method="relayout",
                    args=[{"xaxis.range": [median_density, x_max], "yaxis.range": [median_cv, y_max]}]
                ),
                # Show bottom left quadrant
                dict(
                    label="Stable & Sparse",
                    method="relayout",
                    args=[{"xaxis.range": [0, median_density], "yaxis.range": [0, median_cv]}]
                ),
                # Show top left quadrant
                dict(
                    label="Unstable & Sparse",
                    method="relayout",
                    args=[{"xaxis.range": [0, median_density], "yaxis.range": [median_cv, y_max]}]
                ),
            ]
        )
    ]
)

# Defining the axises
fig.update_xaxes(range=[0, x_max])
fig.update_yaxes(range=[0, y_max])
fig.show()
